# ML-4: Comprehensive Model Evaluation & Failure Pipeline Export

This notebook evaluates all candidate models, compares Cross-Validation metrics, selects the winning classifier based on PR-AUC/Recall/F1 emphasis, tunes decision thresholds, evaluates once on the untouched holdout test set, and exports the final reusable failure pipeline.

## Key Steps:
1. **Ingest Preprocessed Data & Features**.
2. **Cross-Validation Comparison Table**: Evaluate Accuracy, Precision, Recall, F1-Score, ROC-AUC, and PR-AUC.
3. **Model Selection**: Select winner emphasizing minority class failure detection performance.
4. **Threshold Tuning**: Tune optimal probability decision threshold on validation set.
5. **Final Holdout Test Evaluation**: Evaluate the winning model once on the untouched test set.
6. **Export Reusable Failure Pipeline**: Export preprocessor, model, feature schema, threshold, and metrics to `ml/models/final_failure_pipeline.joblib`.

In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, confusion_matrix, 
    precision_recall_curve
)

sys.path.append(os.path.abspath("../src"))
from feature_engineering import DomainFeatureEngineer, build_preprocessing_pipeline

## 1. Data Ingestion & Preprocessing

In [2]:
data_path = "../../data/cleaned_predictive_maintenance.csv"
if not os.path.exists(data_path):
    data_path = "../../data/ai4i2020.csv"

df = pd.read_csv(data_path)

target_col = 'Machine failure'
excluded_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df.columns if c not in excluded_cols and c != target_col]

X = df[feature_cols].copy()
y = df[target_col].copy()

engineer = DomainFeatureEngineer()
X_eng = engineer.transform(X)

categorical_cols = ['Type']
numerical_cols = [c for c in X_eng.columns if c not in categorical_cols]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_eng, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = build_preprocessing_pipeline(categorical_cols, numerical_cols)
X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

feature_names = list(preprocessor.get_feature_names_out())
X_train = pd.DataFrame(X_train_proc, columns=feature_names)
X_test = pd.DataFrame(X_test_proc, columns=feature_names)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (80, 11), Test shape: (20, 11)


## 2. Cross-Validation Metrics Comparison Table

In [4]:
import re

# Clean feature names so they are compatible with XGBoost
X_train = X_train.copy()
X_train.columns = [
    re.sub(r"[\[\]<>]", "_", str(col))
    for col in X_train.columns
]

# Check class distribution
print("Training class distribution:")
print(y_train.value_counts())

# Calculate class weight for XGBoost
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)

scale_pos_weight = neg_count / pos_count

candidate_models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=6,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        max_depth=10,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=5,
        scale_pos_weight=scale_pos_weight,
        learning_rate=0.05,
        random_state=42,
        eval_metric="logloss"
    )
}

# Use a number of folds that does not exceed the minority-class count
n_splits = min(5, int(y_train.value_counts().min()))

print(f"Using {n_splits}-fold Stratified Cross-Validation")

skf = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

scoring = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision"
]

cv_comparison = []
trained_candidates = {}

for name, model in candidate_models.items():

    cv_res = cross_validate(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring=scoring
    )

    cv_comparison.append({
        "Model": name,
        "Accuracy": round(
            np.mean(cv_res["test_accuracy"]), 4
        ),
        "Precision": round(
            np.mean(cv_res["test_precision"]), 4
        ),
        "Recall": round(
            np.mean(cv_res["test_recall"]), 4
        ),
        "F1-Score": round(
            np.mean(cv_res["test_f1"]), 4
        ),
        "ROC-AUC": round(
            np.mean(cv_res["test_roc_auc"]), 4
        ),
        "PR-AUC": round(
            np.mean(cv_res["test_average_precision"]), 4
        )
    })

    model.fit(X_train, y_train)
    trained_candidates[name] = model

comp_df = pd.DataFrame(cv_comparison)

comp_df

Training class distribution:
Machine failure
0    78
1     2
Name: count, dtype: int64
Using 2-fold Stratified Cross-Validation


C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\bingu\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,Logistic Regression,0.975,0.0,0.0,0.0,0.0000,0.0250
1,Decision Tree,0.975,0.0,0.0,0.0,0.5000,0.0250
2,Random Forest,0.975,0.0,0.0,0.0,0.8077,0.1131
3,XGBoost,0.975,0.0,0.0,0.0,0.5000,0.0250


## 3. Decision Threshold Tuning (Maximizing Validation F1-Score)

In [5]:
winning_model_name = comp_df.sort_values(by=['PR-AUC', 'F1-Score', 'ROC-AUC'], ascending=False).iloc[0]['Model']
winner_model = trained_candidates[winning_model_name]

y_train_probs = winner_model.predict_proba(X_train)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, y_train_probs)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
optimal_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5

print(f"Selected Model: {winning_model_name}")
print(f"Optimal Decision Threshold: {optimal_threshold:.4f} (Validation F1: {f1_scores[best_idx]:.4f})")

Selected Model: Random Forest
Optimal Decision Threshold: 1.0000 (Validation F1: 1.0000)


## 4. Final Holdout Test Set Evaluation

In [7]:
# Make sure X_test has the same feature names as X_train
X_test = X_test.copy()
X_test.columns = X_train.columns

# Predict probabilities for the positive class
y_test_probs = winner_model.predict_proba(X_test)[:, 1]

# Apply the optimized classification threshold
y_test_pred_tuned = (
    y_test_probs >= optimal_threshold
).astype(int)

test_metrics = {
    "Accuracy": round(
        accuracy_score(y_test, y_test_pred_tuned), 4
    ),
    "Precision": round(
        precision_score(
            y_test,
            y_test_pred_tuned,
            zero_division=0
        ),
        4
    ),
    "Recall": round(
        recall_score(
            y_test,
            y_test_pred_tuned,
            zero_division=0
        ),
        4
    ),
    "F1-Score": round(
        f1_score(
            y_test,
            y_test_pred_tuned,
            zero_division=0
        ),
        4
    ),
    "ROC-AUC": round(
        roc_auc_score(y_test, y_test_probs), 4
    ),
    "PR-AUC": round(
        average_precision_score(y_test, y_test_probs), 4
    )
}

print(
    "Untouched Test Set Evaluation Metrics "
    "(Tuned Threshold):"
)

for k, v in test_metrics.items():
    print(f"  {k}: {v}")

cm = confusion_matrix(
    y_test,
    y_test_pred_tuned
)

print("\nConfusion Matrix:")
print(cm)

Untouched Test Set Evaluation Metrics (Tuned Threshold):
  Accuracy: 0.95
  Precision: 0.0
  Recall: 0.0
  F1-Score: 0.0
  ROC-AUC: 0.7895
  PR-AUC: 0.2

Confusion Matrix:
[[19  0]
 [ 1  0]]


## 5. Export Reusable Failure Pipeline Bundle

In [8]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

pipeline_bundle = {
    "preprocessor": preprocessor,
    "model": winner_model,
    "winning_model_name": winning_model_name,
    "optimal_threshold": optimal_threshold,
    "feature_names": feature_names,
    "input_feature_schema": feature_cols,
    "test_metrics": test_metrics
}

bundle_path = os.path.join(models_dir, "final_failure_pipeline.joblib")
joblib.dump(pipeline_bundle, bundle_path)
print(f"Final reusable failure pipeline package exported to: {bundle_path}")

Final reusable failure pipeline package exported to: ../models\final_failure_pipeline.joblib


In [9]:
print("Training set:")
print(y_train.value_counts())

print("\nTest set:")
print(y_test.value_counts())

print("\nOptimal threshold:")
print(optimal_threshold)

print("Test probabilities:")
print(y_test_probs)

Training set:
Machine failure
0    78
1     2
Name: count, dtype: int64

Test set:
Machine failure
0    19
1     1
Name: count, dtype: int64

Optimal threshold:
1.0
Test probabilities:
[0.   0.11 0.04 0.   0.02 0.   0.   0.05 0.   0.   0.   0.03 0.   0.
 0.25 0.   0.   0.   0.02 0.  ]
